In [ ]:
import numpy as np
import torch
import cv2
import matplotlib.pyplot as plt
import torchvision.transforms as transforms
import os
import gc
from google.colab import files

In [ ]:
data = np.load(f"/content/nerf.npz")
images_train = data["images_train"].astype(np.float32) / 255.0
c2ws_train = data["c2ws_train"].astype(np.float32)
images_val = data["images_val"].astype(np.float32) / 255.0
c2ws_val = data["c2ws_val"].astype(np.float32)
c2ws_test = data["c2ws_test"].astype(np.float32)
focal = data["focal"].astype(np.float32)

## Part 2.1

In [ ]:

torch.set_default_tensor_type(torch.FloatTensor)

def cartesian_to_homogeneous(points):
    ones = torch.ones((points.shape[0], 1), dtype=points.dtype, device=points.device)
    points_h = torch.cat([points, ones], dim=1)
    return points_h

def homogeneous_to_cartesian(points):
    w = points[:, -1].unsqueeze(1)  
    return points[:, :-1] / w

def transform(c2w, x_c):
    x_c_homo = cartesian_to_homogeneous(x_c)  
    x_w_homo = torch.einsum('nij,nj->ni', c2w, x_c_homo)
    x_w = homogeneous_to_cartesian(x_w_homo)
    return x_w

def pixel_to_camera(K, uv, s):
    uv_homo = cartesian_to_homogeneous(uv)
    K_inv = torch.inverse(K)
    x_c = uv_homo @ K_inv.T

    if torch.is_tensor(s):
        s = s.to(dtype=x_c.dtype, device=x_c.device)
        if s.ndim == 0:
            x_c *= s
        elif s.ndim == 1 and s.shape[0] == uv.shape[0]:
            x_c *= s.unsqueeze(1)
        else:
            raise ValueError("Invalid shape for 's'. Expected scalar or (N,).")
    else:
        x_c *= s 

    return x_c

def pixel_to_ray(K, c2w, uv):
    x_c = pixel_to_camera(K, uv, 1)
    N = x_c.shape[0]
    if c2w.dim() == 2:
        c2w = c2w.unsqueeze(0).expand(N, -1, -1) 
    x_w = transform(c2w, x_c) 
    r_o = c2w[:, :3, 3]       
    r_d = x_w - r_o
    r_d = r_d / torch.norm(r_d, dim=-1, keepdim=True)
    return r_o, r_d

## Part 2.2

In [ ]:
class RaysData(torch.utils.data.Dataset):
    @property
    def K(self):
        return self._K
    
    def len(self):
        return self.height * self.width * self.num_images
    
    def __init__(self, images, c2ws, focal: float, near:float=2.0, far:float=6.0, n_samples=64, device='cuda'):
        if not isinstance(images, torch.Tensor):
            images = torch.from_numpy(images)
        if not isinstance(c2ws, torch.Tensor):
            c2ws = torch.from_numpy(c2ws)

        B, H, W, _ = images.shape
        self.height = H
        self.width = W
        self.num_images = B
        self.n_samples=n_samples
        assert c2ws.shape[0] == B, "The batch dimension must align."
        self.c2ws = c2ws.to(device)
        self.focal = focal
        self.images = images.to(device)
        o_x = W / 2.0
        o_y = H / 2.0
        self._K = torch.tensor([
            [focal, 0.0, o_x],
            [0.0, focal, o_y],
            [0.0, 0.0, 1.0]
        ], dtype=c2ws.dtype, device=device)
        self.near = near
        self.far = far
        self.device = device

    def sample_points_from_rays(self, r_o, r_d, n_samples=64, perturb=True):
        device = r_o.device
        N_rays = r_o.shape[0]

        t_vals = torch.linspace(self.near, self.far, steps=n_samples + 1, device=device)[:-1] 
        t_width = (self.far - self.near) / n_samples
        t_vals = t_vals.expand(N_rays, n_samples) 

        if perturb:
            rand = torch.rand(N_rays, n_samples, device=device)
            t_vals = t_vals + rand * t_width

        t_vals = t_vals.unsqueeze(-1) 
        r_o = r_o.unsqueeze(1) 
        r_d = r_d.unsqueeze(1) 
        sampled_pts = r_o + r_d * t_vals 

        return sampled_pts

    def sample_N(self, N):
        total_rays = self.__len__()
        idx = torch.randperm(total_rays, device=self.device)[:N]
        idx_in_img = idx % (self.height * self.width) 
        img_ids = idx // (self.height * self.width)    
        xs = idx_in_img % self.width                  
        ys = idx_in_img // self.width                 
        xs = xs.long()
        ys = ys.long()

        img_ids = img_ids.long()
        imgs = self.images[img_ids] 
        rgbs = imgs[torch.arange(N), ys, xs] 

        xs, ys = xs + 0.5, ys + 0.5

        pixel_coords = torch.stack([xs, ys], dim=-1)
        c2ws = self.c2ws[img_ids]

        r_o, r_d = pixel_to_ray(self.K, c2ws, pixel_coords)
        sampled_pts = self.sample_points_from_rays(r_o, r_d, n_samples=self.n_samples)
        return r_o, r_d, rgbs, sampled_pts

## Part 2.3

In [ ]:
!pip install viser
import viser, time
# --- You Need to Implement These ------
dataset = RaysData(images=images_train, c2ws=c2ws_train, focal=float(focal)) 
rays_o, rays_d, pixels, points = dataset.sample_N(100) 
H, W = images_train.shape[1:3]
K = dataset.K.cpu().numpy()
# ---------------------------------------

server = viser.ViserServer(share=True)
for i, (image, c2w) in enumerate(zip(images_train, c2ws_train)):
    fov = 2 * np.arctan2(H / 2, K[0, 0])
    server.add_camera_frustum(
        f"/cameras/{i}",
        fov=fov,
        aspect=W / H,
        scale=0.15,
        wxyz=viser.transforms.SO3.from_matrix(c2w[:3, :3]).wxyz,
        position=c2w[:3, 3],
        image=image
    )
for i, (o, d) in enumerate(zip(rays_o, rays_d)):
    o_np = o.cpu().numpy()
    d_np = d.cpu().numpy()
    server.add_spline_catmull_rom(
        f"/rays/{i}", positions=np.stack((o_np, o_np + d_np * 6.0)),
    )

server.add_point_cloud(
    f"/samples",
    colors=np.zeros_like(points.cpu().numpy()).reshape(-1, 3),
    points=points.cpu().numpy().reshape(-1, 3),
    point_size=0.02,
)
time.sleep(0.1)

## Part 2.4

In [ ]:
class PE(torch.nn.Module):
    def __init__(self, L:int =10):
        super(PE, self).__init__()
        self.L = L

    def forward(self, x):
        res = []
        for xx in x.split(1, dim=-1):
            pe  = self._encode_coord(xx)
            res.append(pe)
        return torch.cat(res, dim=-1)

    def _encode_coord(self, x):
        freq = torch.pow(2.0, torch.arange(self.L, device=x.device)) * torch.pi 
        sins = torch.sin(x * freq)
        coss = torch.cos(x * freq)
        return torch.cat([x, sins, coss], dim=-1) 

class Nerf(torch.nn.Module):
    def __init__(self, hidden_dim=512):
        super(Nerf, self).__init__()
        self.pe_pos = PE(L=6)
        self.pe_dir = PE(L=3)
        self.layer1 = torch.nn.Sequential(
            torch.nn.Linear(3*(2*self.pe_pos.L + 1), hidden_dim),
            torch.nn.ReLU(),
            torch.nn.Linear(hidden_dim, hidden_dim),
            torch.nn.ReLU(),
            torch.nn.Linear(hidden_dim, hidden_dim),
            torch.nn.ReLU(),
            torch.nn.Linear(hidden_dim, hidden_dim),
            torch.nn.ReLU(),
        )

        self.layer2 = torch.nn.Sequential(
            torch.nn.Linear(hidden_dim + 3*(2*self.pe_pos.L + 1), hidden_dim),
            torch.nn.ReLU(),
            torch.nn.Linear(hidden_dim, hidden_dim),
            torch.nn.ReLU(),
            torch.nn.Linear(hidden_dim, hidden_dim),
            torch.nn.ReLU(),
            torch.nn.Linear(hidden_dim, hidden_dim),
            torch.nn.ReLU(),
        )

        self.layer_density = torch.nn.Sequential(
            torch.nn.Linear(hidden_dim, 1),
            torch.nn.ReLU(),
        )

        self.layer3 = torch.nn.Linear(hidden_dim, hidden_dim)

        self.layer_rgb = torch.nn.Sequential(
            torch.nn.Linear(hidden_dim + 3*(2*self.pe_dir.L + 1) , hidden_dim // 2),
            torch.nn.ReLU(),
            torch.nn.Linear(hidden_dim // 2, 3),
            torch.nn.Sigmoid()
        )

    def forward(self, x, r_d):
        x_encoded = self.pe_pos(x)
        dir_encoded = self.pe_dir(r_d) 

        h = self.layer1(x_encoded)
        h = self.layer2(torch.cat([h, x_encoded], dim=-1))

        density = self.layer_density(h)

        rgb_feat = self.layer3(h)
        rgb_feat = torch.cat([rgb_feat, dir_encoded], dim=-1)
        rgb = self.layer_rgb(rgb_feat)

        return density, rgb

## Part 2.5

In [ ]:
def volrend(sigmas, rgbs, step_size):
    N_rays, N_samples, _ = sigmas.shape
    alpha = 1.0 - torch.exp(-sigmas * step_size)
    transmittance = torch.ones((N_rays, N_samples, 1), device=sigmas.device)
    transmittance[:, 1:, :] = torch.cumprod(1.0 - alpha[:, :-1, :], dim=1)
    weighted_colors = transmittance * alpha * rgbs  
    integrated_color = torch.sum(weighted_colors, dim=1)  
    return integrated_color

def volrend_color_background(sigmas, rgbs, step_size, background_color=None):
    N_rays, N_samples, _ = sigmas.shape
    if torch.is_tensor(step_size):
        step_size = step_size.view(N_rays, 1, 1)
    else:
        step_size = torch.tensor(step_size, device=sigmas.device).view(1, 1, 1).expand(N_rays, 1, 1)

    alpha = 1.0 - torch.exp(-sigmas * step_size)
    transmittance = torch.ones((N_rays, N_samples, 1), device=sigmas.device)
    transmittance[:, 1:, :] = torch.cumprod(1.0 - alpha[:, :-1, :], dim=1)
    weights = transmittance * alpha
    rendered_color = torch.sum(weights * rgbs, dim=1)

    if background_color is not None:
        if not torch.is_tensor(background_color):
            background_color = torch.tensor(background_color, device=sigmas.device)
        if background_color.ndim == 1:
            background_color = background_color.view(1, 3).expand(N_rays, 3)
        
        accumulated_transmittance = transmittance[:, -1, :]
        rendered_color = rendered_color + accumulated_transmittance * background_color
    
    return rendered_color


In [ ]:
torch.manual_seed(42)
sigmas = torch.rand((10, 64, 1))
rgbs = torch.rand((10, 64, 3))
step_size = (6.0 - 2.0) / 64
rendered_colors = volrend(sigmas, rgbs, step_size)

correct = torch.tensor([
    [0.5006, 0.3728, 0.4728],
    [0.4322, 0.3559, 0.4134],
    [0.4027, 0.4394, 0.4610],
    [0.4514, 0.3829, 0.4196],
    [0.4002, 0.4599, 0.4103],
    [0.4471, 0.4044, 0.4069],
    [0.4285, 0.4072, 0.3777],
    [0.4152, 0.4190, 0.4361],
    [0.4051, 0.3651, 0.3969],
    [0.3253, 0.3587, 0.4215]
  ], dtype=torch.float32)
print(rendered_colors)
assert torch.allclose(rendered_colors, correct, rtol=1e-4, atol=1e-4)

In [ ]:
class PSNR(torch.nn.Module):
    def __init__(self):
        super(PSNR, self).__init__()

    def forward(self, x, y):
        assert x.shape == y.shape, "two images must have the same shape"
        if len(x.shape) == 3:
            x = x.unsqueeze(0)
            y = y.unsqueeze(0)
        B = x.shape[0]
        x = x.view(B, -1)
        y = y.view(B, -1)
        mse = torch.clamp(torch.mean((x - y) ** 2, dim=1), min=1e-5)  
        psnr = 10 * torch.log10(1.0 / mse)  
        return psnr

def reconstruct_images_from_c2ws(model, dataset, c2ws, chunk_size=500, color=None):
    B, H, W = c2ws.shape[0], dataset.height, dataset.width
    device, dtype = c2ws.device, c2ws.dtype
    step_size = (dataset.far - dataset.near) / dataset.n_samples

    y, x = torch.meshgrid(torch.arange(H, dtype=dtype, device=device), 
                         torch.arange(W, dtype=dtype, device=device), indexing='ij')
    pixel_coords = torch.stack([x, y], dim=-1).reshape(-1, 2) + 0.5
    num_rays = H * W
    predicted_rgbs = torch.zeros(B, num_rays, 3, device=device, dtype=torch.float32)

    with torch.no_grad():
        for i in range(0, num_rays, chunk_size):
            actual_chunk_size = min(chunk_size, num_rays - i)
            pixel_chunk = pixel_coords[i:i + actual_chunk_size].unsqueeze(0).expand(B, -1, -1).reshape(B * actual_chunk_size, 2)
            c2ws_chunk = c2ws.unsqueeze(1).expand(-1, actual_chunk_size, -1, -1).reshape(B * actual_chunk_size, 4, 4)
            
            r_o, r_d = pixel_to_ray(dataset.K, c2ws_chunk, pixel_chunk)
            sampled_pts = dataset.sample_points_from_rays(r_o, r_d)
            pred_rgb = predict_rgbs(model, sampled_pts, r_d, step_size, color=color)
            
            predicted_rgbs[:, i:i + actual_chunk_size, :] = pred_rgb.view(B, actual_chunk_size, 3)

    return predicted_rgbs.view(B, H, W, 3)

def predict_rgbs(model, sampled_points, directions, step_size, color=None, chunk_size=2048):
    B, N = sampled_points.shape[:2]
    directions = directions.view(B, 1, 3).repeat(1, N, 1).view(-1, 3)
    sampled_points = sampled_points.view(-1, 3)
    total_points = B * N

    density_chunks, rgb_chunks = zip(*[
        model(sampled_points[i:min(i + chunk_size, total_points)], 
              directions[i:min(i + chunk_size, total_points)])
        for i in range(0, total_points, chunk_size)
    ])

    predicted_density = torch.cat(density_chunks, dim=0).view(B, N, -1)
    predicted_rgbs = torch.cat(rgb_chunks, dim=0).view(B, N, -1)

    return volrend_color_background(predicted_density, predicted_rgbs, step_size, background_color=color)

def get_model(device, path=None):
    model = Nerf()
    model.float()
    if path is not None:
        state_dict = torch.load(path, map_location=device)
        model.load_state_dict(state_dict)
    model.to(device)
    return model

## Part 2.6

In [ ]:
batch = 4096
epochs = 6200
lr = 5e-4
sample_size = 128

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
torch.cuda.empty_cache()
gc.collect()
model_save_path = '/content/final_model.pth'
os.makedirs(os.path.dirname(model_save_path), exist_ok=True)
model = get_model(device)

dataset =  RaysData(images=images_train, c2ws=c2ws_train, focal=float(focal), n_samples=sample_size) #[0][np.newaxis, :]
step_size = (dataset.far - dataset.near) / dataset.n_samples
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

criterion = torch.nn.MSELoss()
reconstruct_criterion = PSNR()
images_val_tensor = torch.from_numpy(images_val).to(dtype=torch.float32, device=device)
c2ws_val_tensor = torch.from_numpy(c2ws_val).to(dtype=torch.float32, device=device)

epoch_losses = []
epoch_reconstruct_psnr = []
reconstructed_pics = dict()
epoch_train_psnr = []
train_batch_accumulator = []

psnr_logging_freq = 200
reconstruct_pic_save_freqs = 1000
train_psnr_freq = 100

for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()

    rays_o, rays_d, rgbs, sampled_points = dataset.sample_N(batch)
    rays_d = rays_d.to(device)
    sampled_points = sampled_points.to(device)
    rgbs = rgbs.to(device)
    rays_o = rays_o.to(device)

    rendered_rgbs = predict_rgbs(model, sampled_points, rays_d, step_size)
    loss = criterion(rendered_rgbs, rgbs)
    loss.backward()
    optimizer.step()
    epoch_losses.append(loss.item())

    if epoch % train_psnr_freq == 0:
        with torch.no_grad():
            train_psnr = reconstruct_criterion(rendered_rgbs, rgbs)
            train_psnr_value = torch.mean(train_psnr).item()
            epoch_train_psnr.append(train_psnr_value)
            print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item()}, Train PSNR: {train_psnr_value:.2f}")
    else:
        print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item()}")

    if epoch % psnr_logging_freq == 0:
        model.eval()
        with torch.no_grad():
            psnr_avg = 0.0
            for val_c2w, val_img in zip(c2ws_val_tensor, images_val_tensor):
                reconstructed_image = reconstruct_images_from_c2ws(model, dataset, val_c2w.unsqueeze(0))
                psnr = reconstruct_criterion(reconstructed_image, val_img.unsqueeze(0))
                psnr_avg += torch.mean(psnr).item()
            psnr_avg /= len(images_val)
            epoch_reconstruct_psnr.append(psnr_avg)
            print(f"Validation PSNR at epoch {epoch+1}: {psnr_avg}")
            if epoch % reconstruct_pic_save_freqs == 0:
                reconstructed_pics[f"reconstructed pic epoch {epoch+1}"] = reconstructed_image.cpu().numpy() #10 reconstructed images
        model.train()

torch.save(model.state_dict(), model_save_path)
files.download('/content/final_model.pth')

In [ ]:
model.eval()
val_img_idx = 0
color = torch.tensor([0.7, 0.85, 1.0], device=device)
reconstructed_image = reconstruct_images_from_c2ws(model, dataset, c2ws_val_tensor[val_img_idx,:].unsqueeze(0), color)

plt.imshow(reconstructed_image.cpu().numpy()[0])
plt.axis('off')
plt.show()

In [ ]:
plt.plot(epoch_losses)
plt.title('Training Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.show()
plt.savefig('Training loss.png')

In [ ]:
plt.figure(figsize=(10, 6))

x_ticks = range(len(epoch_reconstruct_psnr))
x_values = [x * psnr_logging_freq for x in x_ticks]

plt.plot(x_values, epoch_reconstruct_psnr, label='Training Progress')
plt.title('Reconstruction PSNR of A randomly selected validation image')
plt.xlabel('Epochs')
plt.ylabel('PSNR')
plt.legend()
plt.grid(True)
plt.show()
plt.savefig('PSNR.png')

In [ ]:
import math
def look_at_origin(pos, up_axis='y'):
    forward = -pos / np.linalg.norm(pos) 
    if up_axis.lower() == 'x':
        up = np.array([1, 0, 0])
    elif up_axis.lower() == 'y':
        up = np.array([0, 1, 0]) 
    elif up_axis.lower() == 'z':
        up = np.array([0, 0, 1])
    else:
        raise ValueError("up_axis must be 'x', 'y', or 'z'")

    right = np.cross(up, forward)
    right = right / np.linalg.norm(right)
    up = np.cross(forward, right)

    c2w = np.eye(4)
    c2w[:3, 0] = right
    c2w[:3, 1] = up
    c2w[:3, 2] = forward
    c2w[:3, 3] = pos

    return c2w

def rot_x(phi):
    return np.array([
        [1, 0, 0, 0],
        [0, math.cos(phi), -math.sin(phi), 0],
        [0, math.sin(phi), math.cos(phi), 0],
        [0, 0, 0, 1],
    ])

def rot_y(phi):
    return np.array([
        [math.cos(phi), 0, math.sin(phi), 0],
        [0, 1, 0, 0],
        [-math.sin(phi), 0, math.cos(phi), 0],
        [0, 0, 0, 1],
    ])

def rot_z(phi):
    return np.array([
        [math.cos(phi), -math.sin(phi), 0, 0],
        [math.sin(phi), math.cos(phi), 0, 0],
        [0, 0, 1, 0],
        [0, 0, 0, 1],
    ])

def generate_circular_trajectory(start_pos, rotation_axis='x', up_axis='y', num_samples=60,
                                radius_scale=1.0, height_offset=0.0):
    if rotation_axis.lower() == 'x':
        rot_func = rot_x
    elif rotation_axis.lower() == 'y':
        rot_func = rot_y
    elif rotation_axis.lower() == 'z':
        rot_func = rot_z
    else:
        raise ValueError("rotation_axis must be 'x', 'y', or 'z'")

    scaled_start_pos = start_pos * radius_scale
    if rotation_axis.lower() == 'x':
        scaled_start_pos[0] += height_offset
    elif rotation_axis.lower() == 'y':
        scaled_start_pos[1] += height_offset
    elif rotation_axis.lower() == 'z':
        scaled_start_pos[2] += height_offset

    trajectory = []
    for phi in np.linspace(360., 0., num_samples, endpoint=False):
        c2w = look_at_origin(scaled_start_pos, up_axis=up_axis)
        rotated_c2w = rot_func(phi/180.*np.pi) @ c2w
        trajectory.append(rotated_c2w)

    return trajectory

def render_trajectory_frames(model, dataset, trajectory, color=None, chunk_size=500):
    device = next(model.parameters()).device
    model.eval()
    frames = []

    with torch.no_grad():
        for i, c2w in enumerate(trajectory):
            if i % 10 == 0:
                print(f"Rendering frame {i+1}/{len(trajectory)}")

            c2w_tensor = torch.from_numpy(c2w).float().to(device).unsqueeze(0)
            rendered_image = reconstruct_images_from_c2ws(
                model, dataset, c2w_tensor, chunk_size=chunk_size, color=color
            )
            frame = rendered_image[0].cpu().numpy()
            frame = np.clip(frame, 0, 1)
            frames.append(frame)

    return frames

def create_gif_from_frames(frames, output_path, duration=100, loop=0):
    pil_frames = []
    for frame in frames:
        frame_8bit = (frame * 255).astype(np.uint8)
        pil_img = Image.fromarray(frame_8bit)
        pil_frames.append(pil_img)
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    first_frame = pil_frames[0]
    remaining_frames = pil_frames[1:]
    first_frame.save(
        output_path,
        save_all=True,
        append_images=remaining_frames,
        duration=duration,
        loop=loop,
        format='GIF'
    )

    print(f"GIF saved to: {output_path}")

def generate_nerf_gif(model, dataset, output_path='./results/nerf_render.gif',
                     rotation_axis='y', up_axis='y', start_pos=None,
                     num_samples=60, duration=100, color=None, chunk_size=500,
                     radius_scale=1.0, height_offset=0.0):
    if start_pos is None:
        start_pos = np.array([2.5, 0., 0.])

    trajectory = generate_circular_trajectory(
        start_pos=start_pos,
        rotation_axis=rotation_axis,
        up_axis=up_axis,
        num_samples=num_samples,
        radius_scale=radius_scale,
        height_offset=height_offset
    )

    frames = render_trajectory_frames(
        model=model,
        dataset=dataset,
        trajectory=trajectory,
        color=color,
        chunk_size=chunk_size
    )

    create_gif_from_frames(
        frames=frames,
        output_path=output_path,
        duration=duration
    )

    return frames

def generate_standard_gif(model, dataset, rotation_axis='y'):
    device = next(model.parameters()).device
    color = torch.tensor([0.7, 0.85, 1.0], device=device)
    start_positions = {
        'x': np.array([0., 2.5, 0.]),  # Rotate around X 
        'y': np.array([2.5, 0., 0.]),  # Rotate around Y 
        'z': np.array([0., 0., 2.5])   # Rotate around Z
    }

    start_pos = start_positions.get(rotation_axis, np.array([2.5, 0., 0.]))

    generate_nerf_gif(
        model=model,
        dataset=dataset,
        output_path=f'/content/nerf_render_{rotation_axis}_axis.gif',
        rotation_axis=rotation_axis,
        up_axis='z',
        start_pos=start_pos,
        num_samples=60,
        color=color,
        radius_scale=1, 
        duration=100
    )

generate_standard_gif(model, dataset, rotation_axis='z')

In [ ]:
num_images = len(reconstructed_pics)
num_rows = (num_images + 1) // 2  
figsize = (12, 4 * num_rows)

fig, axes = plt.subplots(num_rows, 2, figsize=figsize)
if num_rows == 1:
    axes = axes[None, :]  # Make 2D for consistent indexing
for idx, (title, img) in enumerate(reconstructed_pics.items()):
    row = idx // 2
    col = idx % 2

    ax = axes[row, col]

    if isinstance(img, torch.Tensor):
        img_to_plot = img.squeeze(0).cpu().numpy()
    else:
        img_to_plot = img.squeeze(0) if img.shape[0] == 1 else img 

    img_to_plot = np.clip(img_to_plot, 0, 1)

    ax.imshow(img_to_plot)
    ax.set_title(title)
    ax.axis('off')

if num_images % 2 == 1:
    axes[-1, -1].remove()

plt.tight_layout()
plt.show()
plt.savefig('Render Process.png')